## Chatbot and rag evaluation

Rag is techique that enhance LLM BY  providing them 
relevent  external knowledge. It has become one of the most 
widely used approaches for building LLM applications

This tutorial will show you how to evaluate your RAG application using Langsmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on the datasets
3. How to measure your application's performance using different evaluation metrics

## Overview 

A typical RAG evaluation worflow consists of three main steps :



In [9]:
## Chat bot
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAPI_KEY"] = os.getenv("OPENAPI_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [11]:
## Create the datapoints
from langsmith import Client

client = Client()

## Define the dataset - these are your test data
dataset_name = "Chatbot Evaluation DOCS"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id = dataset.id,
    examples=[
        {
            "inputs": {"question": "What is Langchain?"},
            "outputs": {"answer": "A framework for building LLM applications."}
        },
        {
            "inputs": {"question": "What is Langsmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications."}
        }
    ]
)

{'example_ids': ['650ea559-015b-4fcd-a145-5f4567d31e71',
  '67dd7c17-3660-498d-8d96-8a761f4201ea'],
 'count': 2,
 'as_of': '2026-11-07T19:09:25.760747459Z'}

In [12]:
## Degine metrics LLM AS A JUDGE

import openai
from langsmith import wrappers

openai_client= wrappers.wrap_openai(openai.OpenAI())

eval_instruction = "You are an expert professor specialized in grading student answer to questions."

def correctness(inputs: dict, outputs: dict,reference_outputs: dict ) -> bool:
    user_content = f"""You are grading the following question: 
    
    {inputs['question']}
    Here is the real answer: 
    {reference_outputs['answer']}
    You are grading the following predicted answer: 
    {outputs['response']}
    Response with CORRECT or INCORRECT:
    Grade:
    """

    response= openai_client.completions.create(
        model="",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instruction},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content

    return response == "CORRECT"

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
## Concision - checks wheter the actual output is less than 2x the length of expected result.len

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]< 2 * len(reference_outputs["answer"])))



In [ ]:
## Run Evaluation



## Evaluators or Metrics

- Goal : Measure "how similiar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode : Require a ground truth (reference) answer supplied through a dataset
- Evaluator : Use LLM-as-judge to assess answer correctness.



In [ ]:
from typing_extensions import Annotated, TypedDict

## Correctness Output Schema

class Correctness():

    

_IncompleteInputError: incomplete input (1912289289.py, line 5)